In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
PROJECT_ROOT = Path.cwd().parent

NORM_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "normalized_embeddings"
)

WINDOW_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "windows"
)

WINDOW_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
subjects = sorted(
    [p.name for p in NORM_ROOT.iterdir()]
)

print(len(subjects))

47


In [4]:
WINDOW_SIZE = 30
STRIDE = 1

rows = []

label_map = {
    "alert": 0,
    "low_vigilant": 1,
    "drowsy": 2
}

for subject in tqdm(subjects):

    for state in [
        "alert",
        "low_vigilant",
        "drowsy"
    ]:

        emb = np.load(
            NORM_ROOT
            / subject
            / f"{state}.npy"
        )

        if len(emb) < WINDOW_SIZE:
            continue

        for start in range(
            0,
            len(emb) - WINDOW_SIZE + 1,
            STRIDE
        ):

            rows.append({
                "subject": subject,
                "state": state,
                "label": label_map[state],
                "start_idx": start,
                "embedding_path": str(
                    NORM_ROOT
                    / subject
                    / f"{state}.npy"
                )
            })

  0%|          | 0/47 [00:00<?, ?it/s]

 32%|███▏      | 15/47 [00:00<00:00, 149.39it/s]

 64%|██████▍   | 30/47 [00:00<00:00, 131.10it/s]

 94%|█████████▎| 44/47 [00:00<00:00, 127.87it/s]

100%|██████████| 47/47 [00:00<00:00, 130.15it/s]

In [5]:
WINDOW_SIZE = 30
STRIDE = 1

rows = []

label_map = {
    "alert": 0,
    "low_vigilant": 1,
    "drowsy": 2
}

subjects = sorted(
    [p.name for p in NORM_ROOT.iterdir()]
)

subjects = [
    s for s in subjects
    if s != "46"
]

for subject in tqdm(subjects):

    for state in [
        "alert",
        "low_vigilant",
        "drowsy"
    ]:

        emb_path = (
            NORM_ROOT
            / subject
            / f"{state}.npy"
        )

        emb = np.load(emb_path)

        if len(emb) < WINDOW_SIZE:
            continue

        for start in range(
            0,
            len(emb) - WINDOW_SIZE + 1,
            STRIDE
        ):

            rows.append({
                "subject": subject,
                "label": label_map[state],
                "state": state,
                "start_idx": start,
                "embedding_path": str(emb_path)
            })

  0%|          | 0/47 [00:00<?, ?it/s]

100%|██████████| 47/47 [00:00<00:00, 564.76it/s]

In [6]:
windows_df = pd.DataFrame(rows)

print(len(windows_df))
print(windows_df["label"].value_counts())

83572
label
0    28089
2    27923
1    27560
Name: count, dtype: int64


In [7]:
windows_df.to_csv(
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "windows.csv",
    index=False
)

In [8]:
from sklearn.model_selection import KFold

TEST_SUBJECTS = [
    "01", "06", "11", "16", "21",
    "26", "31", "36", "41"
]

subjects = sorted(
    windows_df["subject"]
    .astype(str)
    .str.zfill(2)
    .unique()
)

subjects = [
    s for s in subjects
    if s not in TEST_SUBJECTS
]

print("Development subjects:", len(subjects))
print(subjects)
print("\nHeld-out subjects:", TEST_SUBJECTS)
print("Held-out count:", len(TEST_SUBJECTS))
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = []

for fold, (train_idx, val_idx) in enumerate(kf.split(subjects)):

    train_subjects = np.array(subjects)[train_idx]
    val_subjects = np.array(subjects)[val_idx]

    folds.append({
        "fold": fold,
        "train_subjects": train_subjects.tolist(),
        "val_subjects": val_subjects.tolist()
    })

    print(
        f"Fold {fold+1}"
    )
    print(
        "Train:",
        len(train_subjects)
    )
    print(
        "Val:",
        len(val_subjects)
    )
    print()

Development subjects: 38
['02', '03', '04', '05', '07', '08', '09', '10', '12', '13', '14', '15', '17', '18', '19', '20', '22', '23', '24', '25', '27', '28', '29', '30', '32', '33', '34', '35', '37', '38', '39', '40', '42', '43', '44', '45', '47', '48']

Held-out subjects: ['01', '06', '11', '16', '21', '26', '31', '36', '41']
Held-out count: 9
Fold 1
Train: 30
Val: 8

Fold 2
Train: 30
Val: 8

Fold 3
Train: 30
Val: 8

Fold 4
Train: 31
Val: 7

Fold 5
Train: 31
Val: 7



In [9]:
import json

with open(
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "folds.json",
    "w"
) as f:
    json.dump(
        folds,
        f,
        indent=4
    )

In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [11]:
from src.lstm_dataset import LSTMDataset

dataset = LSTMDataset(windows_df)

x, y = dataset[0]

print(x.shape)
print(y)

torch.Size([30, 512])
tensor(0)
